# Café Reviews — Data Audit

**Question this notebook answers:**  
*Is the dataset clean, trustworthy, and ready for analysis?*

**Dataset:** `data/processed/reviews_master.csv`  
**Cafés:** Beanlore · Isobel Coffee House · Third Wave Coffee

---

This is a **data quality audit only** — no charts, no descriptive statistics, no business analysis.  
Research questions and visualizations belong in `01_descriptive_analysis.ipynb`.

## 1. Load Data

In [29]:
import pandas as pd

df = pd.read_csv("../data/processed/reviews_master.csv")
print(f"Loaded {len(df):,} rows and {len(df.columns)} columns.")

Loaded 2,308 rows and 22 columns.


## 2. Dataset Overview

Confirm the dataset loaded correctly.

In [30]:
n_rows, n_cols = df.shape

print("DATASET OVERVIEW")
print(f"  Rows:    {n_rows:,}")
print(f"  Columns: {n_cols}")
print(f"  Shape:   ({n_rows:,}, {n_cols})")
print(f"\nColumn list ({n_cols}):")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")

DATASET OVERVIEW
  Rows:    2,308
  Columns: 22
  Shape:   (2,308, 22)

Column list (22):
   1. cafe
   2. review_text
   3. rating
   4. published_at
   5. language
   6. atmosphere
   7. service
   8. noise_level
   9. food_rating
  10. meal_type
  11. vegetarian_offerings
  12. wait_time
  13. price_per_person
  14. parking
  15. is_local_guide
  16. local_guide_level
  17. author_review_count
  18. review_likes
  19. owner_response
  20. year
  21. month
  22. has_owner_response


## 3. Missing Values Audit

Identify which columns have missing data and how much is usable.

In [31]:
missing_count = df.isna().sum()
missing_pct = (missing_count / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    "missing_count": missing_count,
    "missing_pct": missing_pct,
})
missing_df = missing_df.sort_values("missing_pct", ascending=False)

cols_with_missing = (missing_df["missing_count"] > 0).sum()
print(f"Columns with missing values: {cols_with_missing} of {len(df.columns)}")

if cols_with_missing == 0:
    print("No missing values found.")
else:
    print(f"Highest missing rate: {missing_df.index[0]} ({missing_df.iloc[0]['missing_pct']}%)")

missing_df

Columns with missing values: 11 of 22
Highest missing rate: parking (98.87%)


,missing_count,missing_pct
parking,2282,98.87
vegetarian_offerings,2219,96.14
noise_level,2152,93.24
wait_time,2064,89.43
price_per_person,1576,68.28
meal_type,1546,66.98
owner_response,1154,50.00
food_rating,779,33.75
atmosphere,773,33.49
service,754,32.67


## 4. Duplicate Audit

Check for exact row duplicates and duplicate review text.

In [32]:
n_full_row_dupes = df.duplicated().sum()
n_rows_in_dupe_groups = df.duplicated(keep=False).sum()

print("DUPLICATE AUDIT")
print(f"  Full row duplicates:        {n_full_row_dupes:,}")
print(f"  Rows in duplicate groups:    {n_rows_in_dupe_groups:,}")

if n_rows_in_dupe_groups > 0:
    print("\nExample full row duplicates:")
    display(df[df.duplicated(keep=False)].head(5))

if "review_text" in df.columns:
    n_text_dupes = df.duplicated(subset=["review_text"]).sum()
    n_text_dupe_rows = df.duplicated(subset=["review_text"], keep=False).sum()
    print(f"\n  Duplicate review_text rows:  {n_text_dupes:,}")
    print(f"  Rows sharing duplicate text: {n_text_dupe_rows:,}")

    if n_text_dupe_rows > 0:
        print("\nExample duplicate review_text entries:")
        dupe_texts = (
            df.loc[df.duplicated(subset=["review_text"], keep=False), "review_text"]
            .value_counts()
            .head(3)
            .index
        )
        for text in dupe_texts:
            print(f"\n  Text preview: {str(text)[:100]}...")
            cols = [c for c in ["cafe", "rating", "published_at"] if c in df.columns]
            display(df[df["review_text"] == text][cols].head(3))
else:
    n_text_dupes = 0
    print("\n  Column 'review_text' not found — text duplicate check skipped.")

DUPLICATE AUDIT
  Full row duplicates:        0
  Rows in duplicate groups:    0

  Duplicate review_text rows:  2
  Rows sharing duplicate text: 4

Example duplicate review_text entries:

  Text preview: Good food and nice service...


,cafe,rating,published_at
792,Isobel Coffee House,5,2025-10-30 05:41:14.030000+00:00
831,Isobel Coffee House,4,2025-10-14 08:57:13.628000+00:00



  Text preview: Isobel was a took noch ,great environment provided with kind service by the workers….I like it very ...


,cafe,rating,published_at
1210,Isobel Coffee House,5,2023-08-04 07:59:46.460000+00:00
1212,Isobel Coffee House,5,2023-08-04 08:04:47.178000+00:00


## 5. Data Type Audit

Verify columns were imported with the expected types.

In [38]:
dtype_df = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
})

categories = []
for col in df.columns:
    if pd.api.types.is_datetime64_any_dtype(df[col]):
        categories.append("datetime")
    elif pd.api.types.is_numeric_dtype(df[col]):
        categories.append("numeric")
    else:
        categories.append("categorical")

dtype_df["category"] = categories

numeric_cols = dtype_df.loc[dtype_df["category"] == "numeric", "column"].tolist()
categorical_cols = dtype_df.loc[dtype_df["category"] == "categorical", "column"].tolist()
datetime_cols = dtype_df.loc[dtype_df["category"] == "datetime", "column"].tolist()

# Flag date columns stored as strings
for col in ["published_at"]:
    if col in df.columns and not pd.api.types.is_datetime64_any_dtype(df[col]):
        if col not in datetime_cols:
            datetime_cols.append(col)
            dtype_df.loc[dtype_df["column"] == col, "category"] = "datetime (needs conversion)"

print("DATA TYPE AUDIT")
print(f"  Numeric columns ({len(numeric_cols)}):     {numeric_cols}")
print(f"  Categorical columns ({len(categorical_cols)}): {categorical_cols}")
print(f"  Datetime columns ({len(datetime_cols)}):    {datetime_cols}")

dtype_df

DATA TYPE AUDIT
  Numeric columns (11):     ['rating', 'atmosphere', 'service', 'food_rating', 'is_local_guide', 'local_guide_level', 'author_review_count', 'review_likes', 'year', 'month', 'has_owner_response']
  Categorical columns (11): ['cafe', 'review_text', 'published_at', 'language', 'noise_level', 'meal_type', 'vegetarian_offerings', 'wait_time', 'price_per_person', 'parking', 'owner_response']
  Datetime columns (1):    ['published_at']


,column,dtype,category
0,cafe,str,categorical
1,review_text,str,categorical
2,rating,int64,numeric
3,published_at,str,datetime (needs conversion)
4,language,str,categorical
5,atmosphere,float64,numeric
6,service,float64,numeric
7,noise_level,str,categorical
8,food_rating,float64,numeric
9,meal_type,str,categorical


## 6. Priority Variable Coverage Audit

Check whether the variables needed for analysis are present and sufficiently populated.

In [34]:
priority_cols = [
    "rating", "atmosphere", "service", "food_rating",
    "price_per_person", "noise_level", "meal_type", "parking",
    "vegetarian_offerings", "is_local_guide", "has_owner_response",
]

coverage_rows = []

for col in priority_cols:
    if col not in df.columns:
        coverage_rows.append({
            "column": col,
            "present": False,
            "non_null_count": None,
            "coverage_pct": None,
        })
        continue

    non_null = df[col].notna().sum()
    coverage_rows.append({
        "column": col,
        "present": True,
        "non_null_count": non_null,
        "coverage_pct": round(non_null / len(df) * 100, 2),
    })

coverage_df = pd.DataFrame(coverage_rows)

print("PRIORITY VARIABLE COVERAGE")
present = coverage_df[coverage_df["present"] == True]
missing_cols = coverage_df[coverage_df["present"] == False]["column"].tolist()

if missing_cols:
    print(f"  Columns not in dataset: {missing_cols}")

if not present.empty:
    usable = present[present["coverage_pct"] >= 50]["column"].tolist()
    sparse = present[present["coverage_pct"] < 50]["column"].tolist()
    print(f"  Usable (>=50% coverage): {usable if usable else 'none'}")
    print(f"  Sparse (<50% coverage):  {sparse if sparse else 'none'}")

coverage_df

PRIORITY VARIABLE COVERAGE
  Usable (>=50% coverage): ['rating', 'atmosphere', 'service', 'food_rating', 'is_local_guide', 'has_owner_response']
  Sparse (<50% coverage):  ['price_per_person', 'noise_level', 'meal_type', 'parking', 'vegetarian_offerings']


,column,present,non_null_count,coverage_pct
0,rating,True,2308,100.00
1,atmosphere,True,1535,66.51
2,service,True,1554,67.33
3,food_rating,True,1529,66.25
4,price_per_person,True,732,31.72
5,noise_level,True,156,6.76
6,meal_type,True,762,33.02
7,parking,True,26,1.13
8,vegetarian_offerings,True,89,3.86
9,is_local_guide,True,2308,100.00


## 7. Export Audit Summary

Save a record of dataset quality to `data/processed/data_audit_summary.csv`.

In [35]:
summary_rows = []

# Overview
summary_rows.append({"section": "overview", "column": None, "metric": "total_rows", "value": n_rows})
summary_rows.append({"section": "overview", "column": None, "metric": "total_columns", "value": n_cols})

# Missing values
for col in df.columns:
    summary_rows.append({"section": "missing", "column": col, "metric": "missing_count", "value": int(missing_count[col])})
    summary_rows.append({"section": "missing", "column": col, "metric": "missing_pct", "value": float(missing_pct[col])})

# Duplicates
summary_rows.append({"section": "duplicate", "column": None, "metric": "full_row_duplicates", "value": int(n_full_row_dupes)})
summary_rows.append({"section": "duplicate", "column": None, "metric": "rows_in_duplicate_groups", "value": int(n_rows_in_dupe_groups)})
if "review_text" in df.columns:
    summary_rows.append({"section": "duplicate", "column": "review_text", "metric": "duplicate_text_rows", "value": int(n_text_dupes)})

# Data types
for _, row in dtype_df.iterrows():
    summary_rows.append({"section": "dtype", "column": row["column"], "metric": "dtype", "value": row["dtype"]})
    summary_rows.append({"section": "dtype", "column": row["column"], "metric": "category", "value": row["category"]})

# Priority variable coverage
for _, row in coverage_df.iterrows():
    summary_rows.append({"section": "priority_coverage", "column": row["column"], "metric": "present", "value": row["present"]})
    if row["present"]:
        summary_rows.append({"section": "priority_coverage", "column": row["column"], "metric": "non_null_count", "value": int(row["non_null_count"])})
        summary_rows.append({"section": "priority_coverage", "column": row["column"], "metric": "coverage_pct", "value": float(row["coverage_pct"])})

audit_summary = pd.DataFrame(summary_rows)
output_path = "../data/processed/data_audit_summary.csv"
audit_summary.to_csv(output_path, index=False)

print(f"Audit summary saved to: {output_path}")
print(f"Total records exported: {len(audit_summary):,}")
audit_summary.head(10)

Audit summary saved to: ../data/processed/data_audit_summary.csv
Total records exported: 126


,section,column,metric,value
0,overview,NaN,total_rows,2308
1,overview,NaN,total_columns,22
2,missing,cafe,missing_count,0
3,missing,cafe,missing_pct,0.0
4,missing,review_text,missing_count,0
5,missing,review_text,missing_pct,0.0
6,missing,rating,missing_count,0
7,missing,rating,missing_pct,0.0
8,missing,published_at,missing_count,0
9,missing,published_at,missing_pct,0.0
